# X-RAG · Full pipeline (single master notebook)

Run this **cell by cell, top to bottom**. It does the whole project:

1. **Setup** — install deps, load your HuggingFace token, check the GPU.
2. **Demo (no training)** — retrieval + cited generation with off-the-shelf models.
3. **Build training data** — SFT + DPO datasets.
4. **Train** — SFT → DPO → rewriter → NLI head (LoRA on bf16, **A100 recommended**).
5. **Evaluate** — raw-vs-tuned comparison over the full test set.
6. **Demo UI** — a public Gradio link.

**Hardware:** the generator is Llama-3.1-8B in **bf16** (no QLoRA). Training needs an
**A100** (Colab Pro). On a free **T4** the demo sections (2, 6) work, but training (4) will OOM.

**Training runs from scratch:** the trainers don't resume or write intermediate
checkpoints — each run starts clean and saves only the final adapter (a few minutes
each on this dataset). Mount Drive in cell 0.2 so the final adapters survive a disconnect.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0 · Setup

You already extracted the `.rar` here in Colab. These cells point the runtime at the
extracted project, install packages, and load your token from **Colab Secrets**.

### 0.1 · Locate the project root and `cd` into it

In [2]:
# Step 1: Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Install the unrar library (required for .rar files)
!pip install rarfile

# Step 3: Extract the RAR file
import rarfile

# Replace 'My Drive/path/to/' with the actual folder path where your file lives
rar_path = '/content/drive/MyDrive/colab_pipeline.rar'
extract_path = '/content/colab_pipeline/' # Where you want the files to go

with rarfile.RarFile(rar_path) as rf:
    rf.extractall(extract_path)
    print("Extraction complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extraction complete!


In [3]:
import os, glob
from pathlib import Path

# Find the folder that contains the `app/` package (the project root), wherever
# the rar was extracted (/content, /content/colab_pipeline, a Drive path, ...).
def _find_root():
    here = Path.cwd()
    for cand in [here, here / "colab_pipeline", Path("/content/colab_pipeline"),
                 Path("/content")]:
        if (cand / "app" / "schemas.py").is_file():
            return cand
    hits = glob.glob("/content/**/app/schemas.py", recursive=True)
    if hits:
        return Path(hits[0]).parents[1]
    raise SystemExit("Could not find the project root (no app/schemas.py). "
                     "Make sure you extracted the rar.")

ROOT = _find_root()
os.chdir(ROOT)
print("Project root:", ROOT)
print("Top-level entries:", sorted(p.name for p in ROOT.iterdir())[:15])

Project root: /content/colab_pipeline/colab_pipeline
Top-level entries: ['.env.example', 'README.md', '__pycache__', '_build_master_nb.py', 'app', 'config', 'data', 'datasets', 'docs', 'eval', 'models', 'requirements', 'run_full_pipeline.ipynb', 'tests', 'training']


### 0.2 · (Optional, recommended) Mount Drive for checkpoint persistence

Run this if you plan to **train**. It makes training checkpoints + the HF model cache
live on Drive, so a Colab disconnect doesn't lose your progress.
Skip it if you only want the demo.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/xrag/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/xrag/hf_cache', exist_ok=True)
print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


### 0.3 · Install dependencies (3–5 min the first time)

In [5]:
# install dependencies (first run takes 3-5 min)
!pip -q install "pydantic>=2.5" "pydantic-settings>=2.1" pyyaml "fastapi>=0.110" "uvicorn[standard]>=0.27" "gradio>=4.20" "langgraph>=0.2" "ddgs>=0.1.0" "httpx>=0.27" "beautifulsoup4>=4.12" "lxml>=5.0" "trafilatura>=1.9" "redis>=5.0" "pypdf>=4.0" "faiss-cpu>=1.8" "qdrant-client>=1.8" "datasets>=2.18,<3.0.0" "evaluate>=0.4" "scikit-learn>=1.3" "pyngrok>=7.0"
!pip -q install "torch>=2.2" "transformers>=4.43" "accelerate>=0.30" "sentencepiece>=0.2" "tokenizers>=0.20" "FlagEmbedding>=1.2.10" "sentence-transformers>=2.7" "peft>=0.11" "torchao>=0.16.0" "trl>=0.12" "deepspeed>=0.14" "nltk>=3.8" "rouge-score>=0.1.2" "wandb>=0.17" "tensorboard>=2.16" "huggingface-hub>=0.23"
# ---- Fix Colab version conflicts ----
#   torchao 0.10.0 (Colab default) is too old for peft >=0.11
#   trl <=0.11.3 can't import DPOTrainer with transformers >=4.50
!pip -q install --force-reinstall --no-deps "trl>=0.12"
!pip -q install "torchao>=0.16.0"
!pip -q install pypdf nest_asyncio langgraph
# Verify critical versions
import trl, transformers, peft
print(f"trl={trl.__version__}  transformers={transformers.__version__}  peft={peft.__version__}")
print("deps installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 147.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 204.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

trl=1.5.1  transformers=5.9.0  peft=0.19.1
deps installed.


### 0.4 · Load your HuggingFace token + set environment

This reads `HF_TOKEN` from **Colab Secrets** (left sidebar → 🔑 key icon →
`+ Add new secret`, name it `HF_TOKEN`, toggle *Notebook access* on).

In [6]:
import os
from pathlib import Path
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

# Route checkpoints + HF cache to Drive IF it was mounted in 0.2.
if Path('/content/drive/MyDrive/xrag').exists():
    os.environ['XRAG_CHECKPOINTS_DIR'] = '/content/drive/MyDrive/xrag/models'
    os.environ['HF_HOME']              = '/content/drive/MyDrive/xrag/hf_cache'
    print("Drive persistence ON ->", os.environ['XRAG_CHECKPOINTS_DIR'])
else:
    print("Drive NOT mounted: checkpoints go to ./models and die with the session.")

# Pin the embedder + reranker in VRAM on Gradio startup (saves ~5s/query).
os.environ['XRAG_WARMUP'] = '1'
print("token loaded, env set.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Drive persistence ON -> /content/drive/MyDrive/xrag/models
token loaded, env set.


### 0.5 · GPU + import smoke test

In [7]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(cpu)")
from config.settings import settings
print("Generator model:", settings.generator.model_id)
print("Setup OK.")

NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
GPU available: True | NVIDIA RTX PRO 6000 Blackwell Server Edition
Generator model: NousResearch/Meta-Llama-3.1-8B-Instruct
Setup OK.


### 0.6 · Run the unit tests (~1 second) — confirms the package is intact

In [8]:
!python -m unittest discover -s tests 2>&1 | tail -5

..................................................................
----------------------------------------------------------------------
Ran 66 tests in 0.119s

OK


## 1 · Demo with off-the-shelf models (no training yet)

Verifies retrieval + generation + attribution before any fine-tuning. First run
downloads bge-m3 + reranker (~3 GB) and Llama-3.1-8B (~16 GB bf16).

### 1.1 · Retrieval spine

In [9]:
import nest_asyncio; nest_asyncio.apply()
from app.planning.rewriter import rewrite
from app.retrieval.pipeline import retrieve

rw = rewrite("who won the 2022 world cup and who scored the most goals")
print("Sub-queries:", rw.sub_queries)
chunks = retrieve(rw.sub_queries, top_k=10, standalone_query=rw.standalone_query)
print(f"Got {len(chunks)} chunks")
for c in chunks[:3]:
    print(" ce=%.2f" % c.scores.get('ce', 0.0), "|", c.text[:90])

19:38:18 [INFO] app.planning.rewriter :: loading rewriter unsloth/Llama-3.2-3B-Instruct in bf16 (adapter=None)…


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

19:39:14 [INFO] app.retrieval.rerank :: reranked → top-10 (best ce=7.879, worst ce=5.520)
19:39:14 [INFO] app.retrieval.pipeline :: stage=rerank ms=7375.9
Got 10 chunks
 ce=7.88 | From Wikipedia, the free encyclopedia

Association football tournament in Qatar

"2022 Wor
 ce=7.81 | Leading Goal Scorers in FIFA World Cup 2022 (Updated List) – Golden Boot Winners

Kylian M
 ce=6.76 | World Cup final: Lionel Messi named best player as Kylian Mbappe wins Golden Boot

Argenti


### 1.2 · Full pipeline — cited answer + trust scores

In [10]:
from app.orchestrator.engine import run
from app.schemas import QueryRequest

resp = run(QueryRequest(query="who won the 2022 world cup and who scored the most goals",
                        mode="live_web", top_k=10))
print(resp.answer, "\n")
print(f"overall trust {resp.overall_trust:.2f} · trace {resp.trace_id} · {resp.latency_ms} ms")
for c in resp.citations:
    print(f"  [{c.id}] {c.flag} {c.attribution_score:.2f} {c.title} -> {c.url}")

19:39:14 [INFO] app.orchestrator.graph :: [req_f947be60] corrective-RAG start: 'who won the 2022 world cup and who scored the most goals' (max_attempts=3)
19:39:17 [INFO] app.safety.guard :: loading safety classifier meta-llama/Llama-Guard-3-8B…
19:39:17 [WARNING] app.safety.guard :: Llama Guard unavailable (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-Guard-3-8B.
403 Client Error. (Request ID: Root=1-6a2325e5-1dceb94e73545ad47b633832;75cc7e97-66be-40b1-abaa-1d9cf369de2e)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-Guard-3-8B/resolve/main/config.json.
Access to model meta-llama/Llama-Guard-3-8B is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-Guard-3-8B to ask for access.)  -  using rule-based fallback
19:39:17 [INFO] app.orchestrator.graph :: stage=safety_in ms=273.1
19:39:17 [INFO] app.generation.generator :: loading generator NousResear

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

19:40:29 [INFO] app.orchestrator.graph :: stage=attribute ms=6366.1
Argentina won the 2022 FIFA World Cup [8]. 
The top scorer of the 2022 FIFA World Cup was Kylian Mbappe [7]. 

overall trust 0.63 · trace req_f947be60 · 72118 ms
  [7] red 0.26 World Cup Golden Boot winners: Full list of top scorers and more to... -> https://www.sportingnews.com/ca/soccer/news/world-cup-golden-boot-winners-top-scorers-mens-tournament/3b18abee145543f2d41e7888
  [8] green 0.99 FIFA World Cup - Wikipedia -> https://en.wikipedia.org/wiki/FIFA_World_Cup


## 2 · Build training data

`normalize` builds the SFT dataset from ALCE / HAGRID / ExpertQA. `build_dpo_pairs`
makes preference pairs (chat-templated prompts, optional NLI-margin filter).

### 2.1 · SFT dataset

In [11]:
# SFT data = HAGRID + WebGLM-QA (both have answers with inline [n] citations).
# --webglm_max controls how many WebGLM rows to add (more = bigger, more VISIBLE
# raw-vs-tuned behavioural change, but longer training). Set 0 for all ~43k.
!python -m training.data.normalize \
    --out_train data/train/sft.jsonl \
    --out_test  data/test/sft.jsonl \
    --webglm_max 2000
# If real datasets fail to download and you just want a SMOKE run, add:
#   --allow-seed-only

19:40:29 [WARNING] __main__ :: ALCE data dir data/alce missing - skipping (see note in _load_alce: ALCE gold answers have no inline [n], so it is not in the default SFT mix).
19:40:29 [INFO] __main__ :: alce: 0 rows
19:40:30 [INFO] __main__ :: Loading dataset miracl/hagrid (-) from local folder: datasets/miracl_hagrid
19:40:30 [INFO] __main__ :: HAGRID: kept 339 / dropped 1583 (un-attributable or empty)
19:40:30 [INFO] __main__ :: hagrid: 339 rows
19:40:30 [INFO] __main__ :: Local folder not found for THUDM/webglm-qa (-), downloading from Hugging Face…
19:40:37 [INFO] __main__ :: WebGLM-QA: kept 2000 / dropped 0 (cap=2000)
19:40:37 [INFO] __main__ :: webglm: 2000 rows
19:40:37 [INFO] __main__ :: Local folder not found for cmalaviya/expertqa (main), downloading from Hugging Face…
Generating train split: 1545 examples [00:01, 1521.88 examples/s]
19:40:42 [WARNING] __main__ :: ExpertQA not available (An error occurred while generating the dataset)  -  skipping
19:40:42 [INFO] __main__ :: 

### 2.1b · Dataset inspection

Quick sanity check: print the head of every dataset the pipeline uses.

In [12]:
# ──────────────────────────────────────────────────────────────
# Inspect every dataset the pipeline touches
# ──────────────────────────────────────────────────────────────
import json, os
from datasets import load_from_disk

print('=' * 70)
print('  DATASET INSPECTION')
print('=' * 70)

# ── 1. SFT training data (produced by normalize.py) ──────────
for split, path in [('train', 'data/train/sft.jsonl'),
                     ('test',  'data/test/sft.jsonl')]:
    if os.path.exists(path):
        rows = [json.loads(l) for l in open(path, 'r', encoding='utf-8')]
        print(f'\n── SFT {split}: {len(rows)} rows  ({path})')
        for r in rows[:2]:
            print(f"  query:  {r['query'][:90]}")
            print(f"  docs:   {len(r['docs'])} docs")
            if r['docs']:
                print(f"          first title: {r['docs'][0].get('title','')[:60]}")
            print(f"  answer: {r['answer'][:120]}...")
            print()
    else:
        print(f'\n── SFT {split}: FILE MISSING ({path})')

# ── 2. DPO preference pairs ──────────────────────────────────
dpo_path = 'data/train/dpo.jsonl'
if os.path.exists(dpo_path):
    dpo_rows = [json.loads(l) for l in open(dpo_path, 'r', encoding='utf-8')]
    print(f'\n── DPO pairs: {len(dpo_rows)} rows  ({dpo_path})')
    if dpo_rows:
        r = dpo_rows[0]
        prompt  = r.get('prompt', '')[:100]
        chosen  = r.get('chosen', '')[:100]
        rejected = r.get('rejected', '')[:100]
        print(f'  prompt:   {prompt}...')
        print(f'  chosen:   {chosen}...')
        print(f'  rejected: {rejected}...')
else:
    print(f'\n── DPO: FILE MISSING ({dpo_path})')

# ── 3. Local datasets (loaded from datasets/ folder) ──────────
# SFT uses HAGRID (its answers carry inline [n] citations). The NLI head uses
# WICE + nli_fever + ANLI + VitaminC. ALCE / ExpertQA are NOT used: their gold
# answers have no inline [n], so they'd teach the generator to stop citing.
hf_datasets = [
    ('HAGRID (SFT)',   'miracl/hagrid',         None),
    ('WebGLM-QA (SFT)','THUDM/webglm-qa',       None),
    ('WICE (NLI)',     'jon-tow/wice',          'claim'),
    ('FEVER (NLI)',    'pietrolesci/nli_fever', None),
    ('ANLI (NLI)',     'anli',                  None),
    ('VitaminC (NLI)', 'tals/vitaminc',         None),
]

for name, repo, config in hf_datasets:
    try:
        safe_name = repo.replace('/', '_')
        folder_name = f"{safe_name}_{config}" if config else safe_name
        local_path = os.path.join('datasets', folder_name)
        if os.path.exists(local_path):
            ds = load_from_disk(local_path)
            print(f'\n── {name}: loaded from local folder ({local_path})')
            split = ds['train'] if 'train' in ds else next(iter(ds.values()))
            print(f'   rows: {len(split)}')
            print(f'   columns: {list(split.column_names)}')
            row = split[0]
            for k, v in row.items():
                val = str(v)[:100]
                print(f'   {k}: {val}')
        else:
            print(f'\n── {name}: LOCAL FOLDER MISSING ({local_path})')
    except Exception as e:
        print(f'\n── {name}: FAILED ({e})')

# ── 4. Rewriter data (if exists) ─────────────────────────────
rw_path = 'data/train/rewriter.jsonl'
if os.path.exists(rw_path):
    rw_rows = [json.loads(l) for l in open(rw_path, 'r', encoding='utf-8')]
    print(f'\n── Rewriter: {len(rw_rows)} rows  ({rw_path})')
    if rw_rows:
        r = rw_rows[0]
        print(f"  query:     {r.get('query','')}")
        print(f"  history:   {r.get('history',[])}")
        gj = r.get('gold_json','')[:120]
        print(f'  gold_json: {gj}')
else:
    print(f'\n── Rewriter: not yet generated ({rw_path})')

print('\n' + '=' * 70)
print('  INSPECTION COMPLETE')
print('=' * 70)


  DATASET INSPECTION

── SFT train: 2319 rows  (data/train/sft.jsonl)
  query:  How does new life form?
  docs:   5 docs
          first title: 
  answer: The most widely accepted scientific explanation for how new life forms is that all the key molecules of life can form fr...

  query:  Why is second day chili better?
  docs:   5 docs
          first title: 
  answer: Second day chili is better because it allows the flavors to meld together and become more harmonious[1][4], and the hars...


── SFT test: 20 rows  (data/test/sft.jsonl)
  query:  Is vacuum cold?
  docs:   5 docs
          first title: 
  answer: It depends on what is meant by vacuum. If vacuum is defined as a region of space with nothing in it, then it does not ha...

  query:  Why we don’t generate electricity from sound waves while electricity is needed to produce 
  docs:   5 docs
          first title: 
  answer: While sound waves and energy production principles have long been understood, the technology to convert

### 2.2 · DPO preference pairs

In [13]:
!python -m training.data.build_dpo_pairs \
    --in_path  data/train/sft.jsonl \
    --out_path data/train/dpo.jsonl \
    --tokenizer_id NousResearch/Meta-Llama-3.1-8B-Instruct \
    --nli_margin 0.0

19:40:45 [INFO] __main__ :: loading tokenizer NousResearch/Meta-Llama-3.1-8B-Instruct for chat-template DPO prompts…
19:40:50 [INFO] __main__ :: wrote data/train/dpo.jsonl (2319 pairs)


## 3 · Train (A100 · LoRA on bf16)

Each cell trains one piece, **always from scratch** — no resume, no intermediate
`checkpoint-*` dirs. Every run starts from a clean folder (any stale checkpoints
are removed) and writes only the final adapter. Re-running a cell simply retrains
that piece. Each run takes a few minutes on this dataset, so there's nothing to resume. All four use batch_size=1 (OOM-proof) with grad_accum=8 to keep a healthy effective batch.

A tiny helper to point `config.yaml` at each freshly trained adapter:

In [14]:
import yaml, pathlib
_CFG = pathlib.Path('config/config.yaml')

def set_cfg(path_in_yaml: str, value):
    keys = path_in_yaml.split('.')
    # 1) Persist to config.yaml so the training / eval / Gradio SUBPROCESSES
    #    (`!python ...`) pick it up when they freshly load settings.
    cfg = yaml.safe_load(_CFG.read_text())
    node = cfg
    for k in keys[:-1]:
        node = node[k]
    node[keys[-1]] = value
    _CFG.write_text(yaml.safe_dump(cfg, sort_keys=False))
    # 2) ALSO mutate the live settings singleton IN PLACE, so IN-KERNEL calls
    #    (e.g. the 4.1 A/B compare) see the new adapter without a kernel restart.
    #    Reassigning the module attr would create a new object the lazily-
    #    imported references wouldn't see, so we mutate the existing one.
    try:
        from config.settings import settings as _live
        node = _live
        for k in keys[:-1]:
            node = getattr(node, k)
        setattr(node, keys[-1], value)
    except Exception as e:
        print("warn: live settings not updated (kernel restart picks up YAML):", e)
    print(f"config.yaml: {path_in_yaml} = {value}")

In [15]:
# === Free GPU memory before training ===========================================
# Section 1's demo loaded several models (generator, rewriter, reranker, embedder,
# NLI) into THIS notebook kernel and never released them. Each trainer below runs
# as its own `!python` subprocess and needs that VRAM back, or it OOMs. Drop every
# cached model singleton and empty the CUDA allocator. Safe even if you skipped the
# demo (it just frees nothing).
import gc, importlib
import torch

def free_gpu():
    try:
        from app.generation.generator import Generator
        Generator._registry.clear(); Generator._instance = None
    except Exception:
        pass
    for mod, cls_name in [("app.planning.rewriter", "LLMRewriter"),
                          ("app.safety.guard", "_LlamaGuard"),
                          ("app.attribution.scorer", "NLIScorer"),
                          ("app.retrieval.rerank", "Reranker"),
                          ("app.retrieval.embed_retrieve", "M3Embedder")]:
        try:
            cls = getattr(importlib.import_module(mod), cls_name)
            if hasattr(cls, "_registry"):
                cls._registry.clear()
            cls._instance = None
        except Exception:
            pass
    try:
        from app.retrieval.embed_retrieve import clear_embedding_cache
        clear_embedding_cache()
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        free_b, total_b = torch.cuda.mem_get_info()
        print(f"GPU memory freed -> {free_b/1e9:.1f} GB free of {total_b/1e9:.1f} GB")
    else:
        print("no CUDA device; nothing to free")

free_gpu()

GPU memory freed -> 94.9 GB free of 102.0 GB


### 3.1 · SFT the generator (Llama-3.1-8B + LoRA r=16)

In [16]:
!python -m training.sft_generator \
    --train_jsonl data/train/sft.jsonl \
    --eval_jsonl  data/test/sft.jsonl \
    --output_dir  models/sft_generator_lora \
    --epochs 1 \
    --batch_size 1 --grad_accum 8 --lr 2e-4 \
    --max_seq_len 4096 --lora_r 16 --lora_alpha 32

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
19:40:59 [INFO] __main__ :: loading data: data/train/sft.jsonl
19:40:59 [INFO] __main__ :: train=2319  eval=20
19:40:59 [INFO] __main__ :: loading base model NousResearch/Meta-Llama-3.1-8B-Instruct in bf16…
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [03:18<00:00,  1.47it/s]
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
19:44:28 [INFO] __main__ :: response template (4 tokens): [128006, 78191, 128007, 27

In [17]:
set_cfg('generator.adapter_path', 'models/sft_generator_lora')

config.yaml: generator.adapter_path = models/sft_generator_lora


### 3.2 · DPO on top of SFT (faithfulness)

In [18]:
!python -m training.dpo_generator \
    --train_jsonl data/train/dpo.jsonl \
    --output_dir  models/dpo_generator_lora \
    --sft_adapter models/sft_generator_lora \
    --epochs 1 \
    --batch_size 1 --grad_accum 8 --lr 5e-6 --beta 0.1 \
    --max_len 4096

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
19:51:13 [INFO] __main__ :: dpo pairs: 2319
19:51:19 [INFO] __main__ :: loading base in bf16 + SFT adapter as DPO starting policy…
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [01:47<00:00,  2.71it/s]
19:53:23 [WARNING] __main__ :: DPOConfig on this trl/transformers version does not accept ['max_prompt_length', 'save_safetensors'] - using its defaults for those
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 2319/2319 [00:00<00:00, 49305.98 examples/s

In [19]:
set_cfg('generator.adapter_path', 'models/dpo_generator_lora')

config.yaml: generator.adapter_path = models/dpo_generator_lora


### 3.3 · Rewriter (Llama-3.2-3B + LoRA r=8)

In [20]:
# Build real rewriter training data from the SFT queries (bootstrapped gold).
# Without this, train_rewriter falls back to a useless 2-row synthetic smoke test.
!python -m training.data.build_rewriter_data \
    --in_path  data/train/sft.jsonl \
    --out_path data/train/rewriter.jsonl

20:09:36 [INFO] __main__ :: wrote data/train/rewriter.jsonl (2321 rows: 2 history-seed + 2319 from data/train/sft.jsonl)


In [21]:
!python -m training.train_rewriter \
    --train_jsonl data/train/rewriter.jsonl \
    --output_dir  models/rewriter_lora \
    --epochs 2 \
    --batch_size 1 --grad_accum 8 --lr 2e-4 \
    --max_seq_len 1024 --lora_r 8

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
20:09:51 [INFO] __main__ :: rewriter rows: 2321
20:09:51 [INFO] __main__ :: loading rewriter base unsloth/Llama-3.2-3B-Instruct in bf16…
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 254/254 [01:40<00:00,  2.53it/s]
trainable params: 4,587,520 || all params: 3,217,337,344 || trainable%: 0.1426
20:11:40 [INFO] __main__ :: response template (4 tokens): [128006, 78191, 128007, 271]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 2321/2321 [00:00<00:00, 92459.61 e

In [22]:
set_cfg('rewriter.adapter_path', 'models/rewriter_lora')

config.yaml: rewriter.adapter_path = models/rewriter_lora


### 3.4 · NLI head + temperature calibration

In [23]:
!python -m training.train_nli_head \
    --output_dir models/nli_head \
    --epochs 1 --batch_size 1 --grad_accum 8

20:15:40 [INFO] __main__ :: loading NLI model MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli (head-only training)…
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 394/394 [00:14<00:00, 27.45it/s]
20:16:09 [INFO] __main__ :: trainable head params: 3075
20:16:09 [INFO] __main__ :: Loading dataset jon-tow/wice (claim) from local folder: datasets/jon-tow_wice_claim
20:16:09 [INFO] __main__ ::   jon-tow/wice: +1260 rows
20:16:09 [INFO] __main__ :: Local folder not found for pietrolesci/nli_fever (-), downloading from Hugging Face…
Repo card

In [24]:
set_cfg('attribution.head_adapter_path', 'models/nli_head/classifier_head.pt')

config.yaml: attribution.head_adapter_path = models/nli_head/classifier_head.pt


## 4 · Evaluate on the full test set — raw base vs fine-tuned

We score **every** query in `data/test/sft.jsonl` (the whole test split, not a
sample) through the full pipeline, running BOTH the raw base generator and your
fine-tuned (SFT+DPO) adapter on the **same** retrieved chunks with the **same**
NLI scorer — so the delta is purely the effect of fine-tuning. It runs in-kernel
to reuse the models already in VRAM and writes `docs/eval_results.md`.

In [25]:
from eval.harness import load_eval_queries, evaluate_dual

# The ENTIRE test set (every row of data/test/sft.jsonl). Pass max_queries=N for
# a quick subset while iterating.
queries = load_eval_queries("data/test/sft.jsonl")
print(f"Evaluating raw base vs fine-tuned on all {len(queries)} test queries "
      "(this runs the full pipeline per query)…")
summary = evaluate_dual(queries, out_md="docs/eval_results.md",
                        source="data/test/sft.jsonl")

print("\n--- docs/eval_results.md ---\n")
print(open("docs/eval_results.md").read())

20:21:16 [INFO] eval.harness :: loaded 20 eval queries from data/test/sft.jsonl
Evaluating raw base vs fine-tuned on all 20 test queries (this runs the full pipeline per query)…
20:21:16 [INFO] eval.harness :: dual-evaluating 20 queries (raw base vs tuned)…
20:21:16 [INFO] app.orchestrator.dual :: [req_bc224949] DUAL start: 'Is vacuum cold?' mode=live_web
20:21:16 [INFO] app.safety.guard :: loading safety classifier meta-llama/Llama-Guard-3-8B…
20:21:17 [WARNING] app.safety.guard :: Llama Guard unavailable (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-Guard-3-8B.
403 Client Error. (Request ID: Root=1-6a232fbd-01f5da556905ac5a35bcf78e;93e04b9a-b32f-42c5-8c85-3e1698d82a12)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-Guard-3-8B/resolve/main/config.json.
Access to model meta-llama/Llama-Guard-3-8B is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Lla

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


20:21:20 [INFO] app.orchestrator.dual :: stage=rewrite ms=2895.4
20:21:21 [INFO] app.retrieval.search :: ddgs: 10 results for 'Is vacuum cold?'
20:21:21 [INFO] app.retrieval.pipeline :: stage=search ms=1378.6
20:21:21 [WARNING] app.storage.cache :: cache: Redis unreachable (Error 111 connecting to localhost:6379. Connection refused.)  -  using dict-TTL fallback
20:21:21 [INFO] app.retrieval.scrape :: scrape: 10 urls (0 cache hits, 10 to fetch)
20:21:21 [INFO] app.retrieval.scrape :: scrape skip https://physics.stackexchange.com/questions/486500/temperature-of-vacuum-chamber-on-earth (Client error '403 Forbidden' for url 'https://physics.stackexchange.com/questions/486500/temperature-of-vacuum-chamber-on-earth'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403)
20:21:21 [INFO] app.retrieval.scrape :: scrape skip https://www.reddit.com/r/explainlikeimfive/comments/3n374o/eli5_if_space_is_a_vacuum_how_come_its_so_cold/ (Client error '403 Blocked' for

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


20:28:22 [INFO] app.orchestrator.dual :: stage=rewrite ms=485.5
20:28:23 [INFO] app.retrieval.search :: ddgs: 10 results for 'How do CD players read CDs?'
20:28:23 [INFO] app.retrieval.pipeline :: stage=search ms=1228.6
20:28:23 [WARNING] app.storage.cache :: cache: Redis unreachable (Error 111 connecting to localhost:6379. Connection refused.)  -  using dict-TTL fallback
20:28:23 [INFO] app.retrieval.scrape :: scrape: 10 urls (0 cache hits, 10 to fetch)
20:28:23 [INFO] app.retrieval.scrape :: scrape skip /clev?event=StartpageResultClick&sc=maXqNizpm0yCVsCiM1poWU6tu79Eck1QzPfkRzZydI5Xccx4d8H6O3vYxhSHqxrVXD0yDcIhUndNmt&payload={"bdsSessionId":"dbbb8075b957475f892db968031e6633","cheqId":"","countryCode":"NL","deviceType":"mobile","endpoint":"search.serp","hasGoogleAds":false,"page_id":"a9szPEKsIQTtDKUA","queryCategory":"web","segment":"startpage.udog","session_id":"vidcag2wWicTHRlr","surface":"serp-web","transport":"href-request"} (Request URL is missing an 'http://' or 'https://' protoc

Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00,  2.43it/s]

20:28:26 [INFO] app.retrieval.embed_retrieve :: retrieved top-50 candidates via RRF
20:28:26 [INFO] app.retrieval.pipeline :: stage=embed_retrieve ms=923.9


20:28:26 [INFO] app.retrieval.rerank :: reranked → top-10 (best ce=6.578, worst ce=5.480)
20:28:26 [INFO] app.retrieval.pipeline :: stage=rerank ms=163.2
20:28:26 [INFO] app.orchestrator.dual :: stage=retrieve ms=4394.3
20:28:34 [INFO] app.orchestrator.dual :: stage=generate_tuned ms=7217.5
20:28:34 [INFO] app.safety.guard :: loading safety classifier meta-llama/Llama-Guard-3-8B…
20:28:34 [WARNING] app.safety.guard :: Llama Guard unavailable (You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-Guard-3-8B.
403 Client Error. (Request ID: Root=1-6a233172-768d28e40a44fc1c64bcfaa1;9e602c45-6383-4ac2-98f1-91f3a3f0bcaf)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-Guard-3-8B/resolve/main/config.json.
Access to model meta-llama/Llama-Guard-3-8B is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-Guard-3-8B to ask for access.)  -  using rule-based fallback
20

## 5 · Gradio demo (public link)

The last cell prints a `https://xxxxx.gradio.live` URL you can share for 72 hours.
The UI has: live-web / PDF modes, a live process panel, and a *Compare raw vs tuned* toggle.

In [26]:
# Launch the Gradio demo IN THIS KERNEL so it reuses the models already in VRAM
# and we can hand back a Colab proxy URL (gradio.live is usually blocked on Colab).
import asyncio, asyncio.runners, os, sys

# nest_asyncio.apply() (called earlier for retrieval) replaced asyncio.run with a
# version that lacks Python 3.12's `loop_factory` kwarg, which uvicorn (gradio's
# server) now passes -> the server thread crashes ("unexpected keyword argument
# 'loop_factory'") and you then see a misleading "Cannot find empty port".
# Restore the stdlib runner (gradio request handlers run in worker threads and
# don't need the nest_asyncio patch), and repoint any uvicorn alias a previous
# failed launch in this kernel may have already cached.
asyncio.run = asyncio.runners.run
for _name, _mod in list(sys.modules.items()):
    if _name.startswith("uvicorn") and _mod is not None and hasattr(_mod, "asyncio_run"):
        _mod.asyncio_run = asyncio.runners.run

os.environ['XRAG_WARMUP'] = '1'
from ui.app_gradio import build_app

demo = build_app()
# No fixed port (a leftover server from a previous attempt can't cause a
# "port in use" error) and share=False (gradio.live is typically unreachable on
# Colab; we use Colab's own port proxy below). prevent_thread_lock keeps the
# server alive in the background so the next lines can print a URL.
demo.launch(share=True, prevent_thread_lock=True)

try:
    from google.colab.output import eval_js
    url = eval_js(f'google.colab.kernel.proxyPort({demo.server_port})')
    print(f"\n>>> Open the X-RAG demo here: {url}")
except Exception:
    print(f"\n>>> Open the X-RAG demo at {demo.local_url}")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://34b87cb55a32235654.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



>>> Open the X-RAG demo here: https://7860-gpu-g4-s-kkb-euw4a1-icexioato80a-a.europe-west4-1.prod.colab.dev


---
### Done

You now have a shareable cited-answer demo, trained LoRA adapters under `models/`
(on Drive if you mounted it), and an eval baseline in `docs/eval_results.md`.

To publish adapters to HuggingFace:
```python
!python -m training.publish_to_hf --local_dir models/dpo_generator_lora --stage DPO --private
```